## Hawker Centre Inspection: Base Model (TSP)

Let node $1$ be the depot and nodes $2,\ldots,n$ be the hawker centres ($n=8$). Every centre must be visited exactly once, starting and ending at the depot.

- $x_{ij}=1$ if the team travels directly from node $i$ to node $j$.
- $c_{ij}$ is the travel time from node $i$ to node $j$ (minutes, directed: $c_{ij}\neq c_{ji}$ in general).
- $u_i$ is the position of centre $i$ in the tour (MTZ ordering variable).

$$
\min \quad \sum_{i=1}^{n}\sum_{j=1}^{n} c_{ij}x_{ij}
$$

subject to

$$
\sum_{j\neq i} x_{ij} = 1 \quad \forall i \qquad\text{(leave each node once)}
$$

$$
\sum_{i\neq j} x_{ij} = 1 \quad \forall j \qquad\text{(enter each node once)}
$$

$$
u_i - u_j + (n-1)\,x_{ij} \le n-2 \quad \forall i\neq j,\; i,j\in\{2,\ldots,n\} \qquad\text{(MTZ subtour elimination)}
$$

$$
u_1 = 1, \qquad 2 \le u_i \le n \quad i=2,\ldots,n, \qquad x_{ij}\in\{0,1\},\; x_{ii}=0
$$

**Node labels:** the depot is node 1 and the hawker centres are nodes 2 to 8. Python arrays start at 0, so in the code the depot is index 0 and node $k$ is index $k-1$. Routes are printed and saved with the node labels (index + 1).

Inspection durations are not in the objective: every centre is visited and each duration is fixed, so they add a constant to the workday and do not affect which route is optimal.

### Imports

In [ ]:
from gurobipy import *
import itertools
import numpy as np
import os
import pandas as pd

### Load Data

In [ ]:
# Find the repo root (the folder that contains data/processed), whether the notebook
# runs from src/ or from the repo root
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in [here, os.path.dirname(here)]
             if os.path.isdir(os.path.join(p, "data", "processed"))), None)
if ROOT is None:
    raise FileNotFoundError(f"Cannot find data/processed from {here}. Put this notebook in the repo's src/ folder.")
print(f"Repo root: {ROOT}")

DATA_PATH = os.path.join(ROOT, "data", "processed")
nodes_df = pd.read_csv(os.path.join(DATA_PATH, "nodes.csv"))
travel_times_df = pd.read_csv(os.path.join(DATA_PATH, "time_matrix_min.csv"), index_col=0)

travel_time_matrix = travel_times_df.to_numpy()
num_nodes = len(nodes_df)
short_names = nodes_df["short_name"].tolist()
print(f"Number of nodes (including depot): {num_nodes}")

### Functions

In [ ]:
def run_tsp(travel_times, verbose=True):
    """Solve the TSP (depot = node 1, index 0) with MTZ constraints. Returns (route, total_travel_time) with 0-based indices."""
    n = len(travel_times)
    tsp = Model("Hawker Centre Inspection TSP")
    tsp.Params.OutputFlag = 1 if verbose else 0
    tsp.Params.MIPGap = 0   # the best tours differ by only 0.02 min, so prove optimality

    # Whether we travel directly from node i to node j
    x = tsp.addVars(n, n, vtype=GRB.BINARY, name="x")
    # Position of node i in the tour (depot fixed at position 1)
    u = tsp.addVars(n, lb=1, ub=n, name="u")

    tsp.setObjective(
        quicksum(float(travel_times[i, j]) * x[i, j] for i in range(n) for j in range(n)),
        GRB.MINIMIZE
    )

    tsp.addConstrs((x[i, i] == 0 for i in range(n)), "no_self_loop")
    tsp.addConstrs((quicksum(x[i, j] for j in range(n)) == 1 for i in range(n)), "leave_once")
    tsp.addConstrs((quicksum(x[i, j] for i in range(n)) == 1 for j in range(n)), "enter_once")

    # MTZ subtour elimination
    tsp.addConstr(u[0] == 1, "depot_position")
    tsp.addConstrs((u[i] >= 2 for i in range(1, n)), "u_lb")
    tsp.addConstrs(
        (u[i] - u[j] + (n - 1) * x[i, j] <= n - 2
         for i in range(1, n) for j in range(1, n) if i != j),
        "mtz"
    )

    tsp.optimize()
    if tsp.Status != GRB.OPTIMAL:
        return None

    # Follow the arcs from the depot to recover the route
    route = [0]
    while True:
        nxt = next(j for j in range(n) if x[route[-1], j].X > 0.5)
        route.append(nxt)
        if nxt == 0:
            break
    return route, route_time(travel_times, route)


def to_labels(route):
    """Convert 0-based indices to node labels (depot = node 1, centres = nodes 2-8)."""
    return [int(i) + 1 for i in route]


def route_time(travel_times, route):
    """Total travel time of a given route under a given travel-time matrix."""
    return sum(travel_times[route[k], route[k + 1]] for k in range(len(route) - 1))


def brute_force_tsp(travel_times):
    """Enumerate all (n-1)! tours. Only for checking the MIP on small instances."""
    n = len(travel_times)
    tours = [(0,) + p + (0,) for p in itertools.permutations(range(1, n))]
    costs = [route_time(travel_times, r) for r in tours]
    order = np.argsort(costs)
    return [(list(tours[k]), costs[k]) for k in order]

### Solve the Base Case

In [ ]:
base_route, base_time = run_tsp(travel_time_matrix)

print(f"Optimal route: {to_labels(base_route)}")
print(" -> ".join(short_names[i] for i in base_route))
print(f"Total travel time: {base_time:.2f} min")

### Check Against Brute Force

In [ ]:
ranked = brute_force_tsp(travel_time_matrix)
print(f"Brute-force optimum: {to_labels(ranked[0][0])}, {ranked[0][1]:.2f} min")
assert abs(ranked[0][1] - base_time) < 1e-6

print("\nTop 5 tours:")
for r, c in ranked[:5]:
    print(f"  {c:6.2f} min  {to_labels(r)}")

reverse_route = base_route[::-1]
print(f"\nSame tour driven in reverse: {route_time(travel_time_matrix, reverse_route):.2f} min")

### Save Results

In [ ]:
RESULTS_PATH = os.path.join(ROOT, "results")
os.makedirs(RESULTS_PATH, exist_ok=True)
pd.DataFrame([{"route": to_labels(base_route), "total_travel_time_min": base_time}]).to_csv(
    os.path.join(RESULTS_PATH, "base_tsp.csv"), index=False
)